In [1]:
# ============================================================
# CELL 1: Installs, Imports, Coordinate Attention
# ============================================================
!pip install ultralytics kagglehub -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import os, sys, shutil, yaml, cv2, math, random, importlib, glob
import numpy as np
from pathlib import Path
from collections import defaultdict
from ultralytics.nn.modules.conv import Conv

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Working directory (local machine) ──
WORK_DIR = os.path.join(os.path.expanduser("~"), "sfd_yolov8_novel")
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Work dir: {WORK_DIR}")

CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]
NC = len(CLASS_NAMES)


class CoordinateAttention(nn.Module):
    """
    Coordinate Attention — encodes spatial position along H and W
    separately, then recalibrates channel responses.
    """
    def __init__(self, in_channels, reduction=32):
        super().__init__()
        mid = max(8, in_channels // reduction)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv_shared = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
        )
        self.conv_h = nn.Conv2d(mid, in_channels, 1)
        self.conv_w = nn.Conv2d(mid, in_channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = self.pool_h(x)                            # (B,C,H,1)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (B,C,W,1)
        combined = torch.cat([x_h, x_w], dim=2)         # (B,C,H+W,1)
        combined = self.conv_shared(combined)
        h_part, w_part = torch.split(combined, [H, W], dim=2)
        w_part = w_part.permute(0, 1, 3, 2)
        att_h = self.conv_h(h_part).sigmoid()
        att_w = self.conv_w(w_part).sigmoid()
        return x * att_h * att_w


# Test
t = torch.randn(1, 64, 80, 80)
assert CoordinateAttention(64)(t).shape == t.shape
print("✓ CoordinateAttention OK")

PyTorch : 2.6.0+cu124
CUDA    : True
GPU     : NVIDIA RTX 2000 Ada Generation
VRAM    : 17.2 GB
Work dir: C:\Users\CSE_SDPL\sfd_yolov8_novel
✓ CoordinateAttention OK


In [2]:
# ============================================================
# CELL 2: DWR_CA + C2f_DWR  (backbone replacement for C2f)
# ============================================================

class DWR_CA(nn.Module):
    """
    Dilation-wise Residual with Coordinate Attention.
    Expands to 3c → three dilated DW convolutions (d=1,3,5)
    → concat → compress → CA → residual add.
    """
    def __init__(self, c):
        super().__init__()
        self.conv_expand = nn.Sequential(
            nn.Conv2d(c, c * 3, 1, bias=False),
            nn.BatchNorm2d(c * 3),
            nn.SiLU(inplace=True),
        )
        self.dw1 = nn.Conv2d(c, c, 3, padding=1,  dilation=1, groups=c, bias=False)
        self.dw2 = nn.Conv2d(c, c, 3, padding=3,  dilation=3, groups=c, bias=False)
        self.dw3 = nn.Conv2d(c, c, 3, padding=5,  dilation=5, groups=c, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.bn2 = nn.BatchNorm2d(c)
        self.bn3 = nn.BatchNorm2d(c)
        self.conv_compress = nn.Sequential(
            nn.Conv2d(c * 3, c, 1, bias=False),
            nn.BatchNorm2d(c),
        )
        self.ca = CoordinateAttention(c)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        expanded = self.conv_expand(x)
        x1, x2, x3 = expanded.chunk(3, dim=1)
        x1 = self.bn1(self.dw1(x1))
        x2 = self.bn2(self.dw2(x2))
        x3 = self.bn3(self.dw3(x3))
        merged = torch.cat([x1, x2, x3], dim=1)
        compressed = self.conv_compress(merged)
        attended = self.ca(compressed)
        return self.act(attended + x)


class C2f_DWR(nn.Module):
    """C2f with DWR_CA blocks replacing standard Bottleneck."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(DWR_CA(self.c) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


# Test
t = torch.randn(1, 64, 80, 80)
assert C2f_DWR(64, 64, n=1)(t).shape == (1, 64, 80, 80)
print("✓ C2f_DWR OK")

✓ C2f_DWR OK


In [3]:
# ============================================================
# CELL 3: Partial Conv + FasterBlock + C2f_FasterBlock
#          (neck replacement for C2f — lightweight)
# ============================================================

class PConv(nn.Module):
    """Partial Convolution — only first 1/4 channels are convolved."""
    def __init__(self, c, k=3):
        super().__init__()
        self.partial_c = c // 4
        self.conv = nn.Conv2d(self.partial_c, self.partial_c, k,
                              padding=k // 2, bias=False)
        self.bn = nn.BatchNorm2d(self.partial_c)

    def forward(self, x):
        x1, x2 = x[:, :self.partial_c], x[:, self.partial_c:]
        x1 = self.bn(self.conv(x1))
        return torch.cat([x1, x2], dim=1)


class FasterBlock(nn.Module):
    """FasterNet block: PConv → expand → BN+SiLU → project + residual."""
    def __init__(self, c):
        super().__init__()
        self.pconv = PConv(c)
        self.conv1 = nn.Conv2d(c, c * 2, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(c * 2)
        self.act   = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(c * 2, c, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(c)

    def forward(self, x):
        out = self.pconv(x)
        out = self.act(self.bn1(self.conv1(out)))
        out = self.bn2(self.conv2(out))
        return out + x


class C2f_FasterBlock(nn.Module):
    """C2f with FasterBlock replacing standard Bottleneck."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(FasterBlock(self.c) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


# Test
t = torch.randn(1, 128, 40, 40)
assert C2f_FasterBlock(128, 128, n=1)(t).shape == (1, 128, 40, 40)
print("✓ C2f_FasterBlock OK")

✓ C2f_FasterBlock OK


In [4]:
# ============================================================
# CELL 4: BiFPNConcat — learnable weighted fusion
#          Drop-in replacement for ultralytics Concat
# ============================================================

class BiFPNConcat(nn.Module):
    """
    BiFPN fast normalized weighted concatenation.
    w_i = ReLU(w_i) / (Σ ReLU(w_j) + ε)
    Each input scale gets a LEARNABLE importance weight.
    """
    def __init__(self, dimension=1):
        super().__init__()
        self.d = dimension
        self.weight = nn.Parameter(torch.ones(4, dtype=torch.float32))
        self.eps = 1e-4

    def forward(self, x):
        n = len(x)
        w = F.relu(self.weight[:n])
        w = w / (w.sum() + self.eps)
        weighted = [w[i] * x[i] for i in range(n)]
        return torch.cat(weighted, self.d)


# Test
t1, t2 = torch.randn(1, 64, 80, 80), torch.randn(1, 128, 80, 80)
m = BiFPNConcat(dimension=1)
assert m([t1, t2]).shape == (1, 192, 80, 80)
print("✓ BiFPNConcat OK")

✓ BiFPNConcat OK


In [5]:
# ============================================================
# CELL 5: Wise-IoU v3 Loss
#
# Non-monotonic focusing:
#   β = loss_i / mean(loss)
#   f(β) = β × exp(1−β)   → max at β=1, suppresses outliers
#
# Improvement: returns quality score so that ultralytics
# computes  loss = 1 − quality = focusing × base_loss
# ============================================================

class WiseIoUv3Loss(nn.Module):
    """Standalone WIoU v3 loss (for testing / direct use)."""
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        b1_x1, b1_y1, b1_x2, b1_y2 = pred.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = target.chunk(4, -1)

        w1 = (b1_x2 - b1_x1).clamp(0); h1 = (b1_y2 - b1_y1).clamp(0)
        w2 = (b2_x2 - b2_x1).clamp(0); h2 = (b2_y2 - b2_y1).clamp(0)

        inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp_(0) * \
                (b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)).clamp_(0)
        union = w1 * h1 + w2 * h2 - inter + self.eps
        iou = inter / union

        cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)
        ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)
        c2 = cw ** 2 + ch ** 2 + self.eps
        rho2 = ((b1_x1 + b1_x2 - b2_x1 - b2_x2) ** 2 +
                (b1_y1 + b1_y2 - b2_y1 - b2_y2) ** 2) / 4
        rho_w = (w1 - w2) ** 2 / (cw ** 2 + self.eps)
        rho_h = (h1 - h2) ** 2 / (ch ** 2 + self.eps)

        base_loss = 1.0 - iou + rho2 / c2 + rho_w + rho_h

        with torch.no_grad():
            mean_loss = base_loss.detach().mean().clamp(min=self.eps)
            beta = base_loss.detach() / mean_loss
            focusing = (beta * torch.exp(1.0 - beta)).clamp(min=self.eps, max=2.0)

        return (focusing * base_loss).mean()


# Test
loss_fn = WiseIoUv3Loss()
p = torch.tensor([[10, 10, 50, 50], [20, 20, 60, 60]], dtype=torch.float32)
g = torch.tensor([[12, 12, 48, 48], [22, 18, 58, 62]], dtype=torch.float32)
print(f"✓ Wise-IoU v3 Loss: {loss_fn(p, g).item():.4f}")

✓ Wise-IoU v3 Loss: 0.2011


In [6]:
# ============================================================
# CELL 6: Download VisDrone via kagglehub + Verify + Prepare
# ============================================================

import kagglehub

print("=" * 60)
print("DATASET DOWNLOAD & SETUP")
print("=" * 60)

# ── Step 1: Download ──
print("\n  Downloading VisDrone dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("banuprasadb/visdrone-dataset")
dataset_path = Path(dataset_path)
print(f"  ✓ Downloaded to: {dataset_path}")

# ── Step 2: Find root ──
def find_visdrone_root(base):
    base = Path(base)
    # Direct check
    if (base / "VisDrone2019-DET-train").exists():
        return base
    # One level down
    for child in base.iterdir():
        if child.is_dir() and (child / "VisDrone2019-DET-train").exists():
            return child
    # Recursive search
    for root, dirs, _ in os.walk(str(base)):
        if Path(root).relative_to(base).parts.__len__() > 5:
            dirs.clear(); continue
        if "VisDrone2019-DET-train" in dirs:
            return Path(root)
    raise FileNotFoundError(f"VisDrone not found under {base}")


VISDRONE_ROOT = find_visdrone_root(dataset_path)
print(f"  ✓ VisDrone root: {VISDRONE_ROOT}")

# ── Step 3: Resolve image/label paths ──
SRC_TRAIN_IMG = VISDRONE_ROOT / "VisDrone2019-DET-train" / "images"
SRC_VAL_IMG   = VISDRONE_ROOT / "VisDrone2019-DET-val"   / "images"

# Labels might be in 'labels' or 'annotations'
def find_label_dir(split_dir):
    for name in ["labels", "annotations", "Labels", "Annotations"]:
        p = split_dir / name
        if p.exists() and len(list(p.glob("*.txt"))) > 0:
            return p
    return split_dir / "labels"  # fallback

SRC_TRAIN_LBL = find_label_dir(VISDRONE_ROOT / "VisDrone2019-DET-train")
SRC_VAL_LBL   = find_label_dir(VISDRONE_ROOT / "VisDrone2019-DET-val")

# ── Step 4: Detect format & convert if VisDrone annotation format ──
def is_visdrone_format(lbl_dir, n_check=5):
    """Check if labels are comma-separated VisDrone format."""
    for f in sorted(Path(lbl_dir).glob("*.txt"))[:n_check]:
        with open(f) as fh:
            line = fh.readline().strip()
            if line and ',' in line and len(line.split(',')) >= 8:
                return True
    return False

def convert_visdrone_to_yolo(src_lbl, src_img, dst_lbl):
    """Convert VisDrone annotations to YOLO format."""
    dst_lbl = Path(dst_lbl)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    converted = 0
    for ann_file in sorted(Path(src_lbl).glob("*.txt")):
        # Get image dimensions
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
            candidate = Path(src_img) / (ann_file.stem + ext)
            if candidate.exists():
                img_path = candidate; break
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        yolo_lines = []
        with open(ann_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(',')
                if len(parts) < 8:
                    continue
                x_left, y_top, bw, bh = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                category = int(parts[5])
                # Skip ignored (0) and others (11)
                if category < 1 or category > 10:
                    continue
                cls_id = category - 1  # 1-based → 0-based
                cx = (x_left + bw / 2) / w
                cy = (y_top + bh / 2) / h
                nw = bw / w
                nh = bh / h
                # Bounds check
                if nw <= 0 or nh <= 0:
                    continue
                yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        with open(dst_lbl / ann_file.name, 'w') as f:
            f.write('\n'.join(yolo_lines) + '\n' if yolo_lines else '')
        converted += 1
    return converted


# Check train labels
NEED_CONVERT = is_visdrone_format(SRC_TRAIN_LBL)
if NEED_CONVERT:
    print("\n  ⚠ Labels are in VisDrone annotation format → converting to YOLO...")

    yolo_train_lbl = Path(WORK_DIR) / "yolo_labels" / "train"
    yolo_val_lbl   = Path(WORK_DIR) / "yolo_labels" / "val"

    n1 = convert_visdrone_to_yolo(SRC_TRAIN_LBL, SRC_TRAIN_IMG, yolo_train_lbl)
    n2 = convert_visdrone_to_yolo(SRC_VAL_LBL, SRC_VAL_IMG, yolo_val_lbl)
    print(f"    ✓ Converted {n1} train + {n2} val annotation files to YOLO format")

    SRC_TRAIN_LBL = yolo_train_lbl
    SRC_VAL_LBL   = yolo_val_lbl
else:
    print("\n  ✓ Labels are already in YOLO format")


# ── Step 5: Print summary ──
for name, path in [("Train images", SRC_TRAIN_IMG), ("Train labels", SRC_TRAIN_LBL),
                    ("Val images",   SRC_VAL_IMG),   ("Val labels",   SRC_VAL_LBL)]:
    n = len(list(path.glob("*"))) if path.exists() else 0
    status = "✓" if path.exists() and n > 0 else "✗"
    print(f"    {name:14s}: {status} {n:5d} files  →  {path}")

# Sample label check
sample = sorted(list(SRC_TRAIN_LBL.glob("*.txt")))[:1]
if sample:
    with open(sample[0]) as f:
        lines = f.readlines()[:3]
    print(f"\n  Sample label ({sample[0].name}):")
    for l in lines:
        print(f"    {l.strip()}")

print(f"\n  ✅ Dataset ready! ({NC} classes)")
print("=" * 60)

C:\Users\CSE_SDPL\anaconda3\envs\tf_gpu\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


DATASET DOWNLOAD & SETUP

  ✓ Downloaded to: C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1
  ✓ VisDrone root: C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1\VisDrone_Dataset

  ✓ Labels are already in YOLO format
    Train images  : ✓  6471 files  →  C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1\VisDrone_Dataset\VisDrone2019-DET-train\images
    Train labels  : ✓  6471 files  →  C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1\VisDrone_Dataset\VisDrone2019-DET-train\labels
    Val images    : ✓   548 files  →  C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1\VisDrone_Dataset\VisDrone2019-DET-val\images
    Val labels    : ✓   548 files  →  C:\Users\CSE_SDPL\.cache\kagglehub\datasets\banuprasadb\visdrone-dataset\versions\1\VisDrone_Dataset\VisDrone2019-DET-val\labels

  Sample label (0000002_00005_d_0000014

In [7]:
# ============================================================
# CELL 7: Copy-Paste Augmentation for Rare Classes
# ============================================================

random.seed(42)
np.random.seed(42)

def count_class_instances(lbl_dir):
    counts = defaultdict(int)
    for lbl_file in Path(lbl_dir).glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    counts[int(parts[0])] += 1
    return dict(counts)


def build_instance_bank(img_dir, lbl_dir, rare_classes, max_per_class=400):
    bank = defaultdict(list)
    for lbl_file in sorted(Path(lbl_dir).glob("*.txt")):
        with open(lbl_file) as f:
            lines = [l.strip() for l in f if l.strip()]
        entries = []
        for line in lines:
            parts = line.split()
            if len(parts) < 5:
                continue
            cid = int(parts[0])
            if cid in rare_classes and len(bank[cid]) < max_per_class:
                entries.append((cid, [float(x) for x in parts[1:5]]))
        if not entries:
            continue
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
            c = Path(img_dir) / (lbl_file.stem + ext)
            if c.exists():
                img_path = c; break
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        for cid, (cx, cy, bw, bh) in entries:
            x1 = max(0, int((cx - bw / 2) * w))
            y1 = max(0, int((cy - bh / 2) * h))
            x2 = min(w, int((cx + bw / 2) * w))
            y2 = min(h, int((cy + bh / 2) * h))
            if x2 - x1 < 10 or y2 - y1 < 10:
                continue
            bank[cid].append({'crop': img[y1:y2, x1:x2].copy()})
    return dict(bank)


def iou_check(box, existing, thr=0.2):
    for eb in existing:
        ix1, iy1 = max(box[0], eb[0]), max(box[1], eb[1])
        ix2, iy2 = min(box[2], eb[2]), min(box[3], eb[3])
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        a1 = (box[2] - box[0]) * (box[3] - box[1])
        a2 = (eb[2] - eb[0]) * (eb[3] - eb[1])
        if inter / (a1 + a2 - inter + 1e-6) > thr:
            return True
    return False


def create_augmented_images(img_dir, lbl_dir, bank, rare_ids,
                             out_img, out_lbl, n_aug=2500):
    out_img, out_lbl = Path(out_img), Path(out_lbl)
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(list(Path(img_dir).glob("*.jpg")) +
                      list(Path(img_dir).glob("*.png")) +
                      list(Path(img_dir).glob("*.JPG")))
    created = 0
    for i in range(n_aug):
        src = all_imgs[np.random.randint(len(all_imgs))]
        lbl = Path(lbl_dir) / (src.stem + ".txt")
        img = cv2.imread(str(src))
        if img is None:
            continue
        h, w = img.shape[:2]
        labels, boxes = [], []
        if lbl.exists():
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        labels.append(line.strip())
                        cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                        boxes.append([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2])

        n_paste = np.random.randint(2, 5)
        pasted = 0
        for _ in range(n_paste * 4):
            if pasted >= n_paste:
                break
            cls = rare_ids[np.random.randint(len(rare_ids))]
            if cls not in bank or not bank[cls]:
                continue
            inst = bank[cls][np.random.randint(len(bank[cls]))]
            crop = inst['crop']
            ch_c, cw_c = crop.shape[:2]
            scale = np.random.uniform(0.6, 1.4)
            nw, nh = max(10, int(cw_c * scale)), max(10, int(ch_c * scale))
            if nw >= w - 10 or nh >= h - 10:
                continue
            px = np.random.randint(5, w - nw - 5)
            py = np.random.randint(5, h - nh - 5)
            nbox = [px / w, py / h, (px + nw) / w, (py + nh) / h]
            if iou_check(nbox, boxes):
                continue
            resized = cv2.resize(crop, (nw, nh))
            # Color jitter
            if np.random.random() > 0.4:
                hsv = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV).astype(np.float32)
                hsv[:, :, 0] = (hsv[:, :, 0] + np.random.uniform(-8, 8)) % 180
                hsv[:, :, 1] *= np.random.uniform(0.8, 1.2)
                hsv[:, :, 2] *= np.random.uniform(0.8, 1.2)
                resized = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)
            # Random horizontal flip
            if np.random.random() > 0.5:
                resized = cv2.flip(resized, 1)
            img[py:py + nh, px:px + nw] = resized
            ncx = (nbox[0] + nbox[2]) / 2
            ncy = (nbox[1] + nbox[3]) / 2
            nbw = nbox[2] - nbox[0]
            nbh = nbox[3] - nbox[1]
            labels.append(f"{cls} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}")
            boxes.append(nbox)
            pasted += 1

        if pasted == 0:
            continue
        name = f"cpaste_{i:05d}"
        cv2.imwrite(str(out_img / f"{name}.jpg"), img)
        with open(out_lbl / f"{name}.txt", 'w') as f:
            f.write('\n'.join(labels) + '\n')
        created += 1
        if (i + 1) % 500 == 0:
            print(f"    Progress: {created} augmented images...")
    return created


# ═══════════════════════════════════════════
# RUN AUGMENTATION PIPELINE
# ═══════════════════════════════════════════
print("=" * 60)
print("COPY-PASTE AUGMENTATION")
print("=" * 60)

counts = count_class_instances(SRC_TRAIN_LBL)
total = sum(counts.values())

print(f"\n  BEFORE augmentation ({total} total instances):")
for cid in sorted(counts.keys()):
    name = CLASS_NAMES[cid] if cid < NC else f"cls_{cid}"
    pct = counts[cid] / total * 100
    bar = "█" * int(pct * 2)
    print(f"    {cid:2d} {name:20s}: {counts[cid]:6d} ({pct:5.1f}%) {bar}")

# Find rare classes (< 5%)
rare = {cid: cnt for cid, cnt in counts.items() if cnt / total * 100 < 5.0}
print(f"\n  Rare classes (< 5%): {[CLASS_NAMES[c] for c in rare if c < NC]}")

# Build crop bank
print("  Building instance bank...")
bank = build_instance_bank(SRC_TRAIN_IMG, SRC_TRAIN_LBL, rare)
for cid in sorted(bank.keys()):
    print(f"    {CLASS_NAMES[cid] if cid < NC else cid:20s}: {len(bank[cid])} crops")

# ── Setup combined training directory ──
DATASET_DIR = os.path.join(WORK_DIR, "dataset")
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

combined_train_img = Path(DATASET_DIR) / "train" / "images"
combined_train_lbl = Path(DATASET_DIR) / "train" / "labels"
combined_train_img.mkdir(parents=True, exist_ok=True)
combined_train_lbl.mkdir(parents=True, exist_ok=True)

# Copy originals (use copy instead of symlink for cross-filesystem compat)
print(f"\n  Copying original training files...")
orig_count = 0
for f in SRC_TRAIN_IMG.glob("*"):
    dst = combined_train_img / f.name
    if not dst.exists():
        try:
            os.symlink(str(f.resolve()), str(dst))
        except OSError:
            shutil.copy2(str(f), str(dst))
        orig_count += 1

for f in SRC_TRAIN_LBL.glob("*.txt"):
    dst = combined_train_lbl / f.name
    if not dst.exists():
        try:
            os.symlink(str(f.resolve()), str(dst))
        except OSError:
            shutil.copy2(str(f), str(dst))

print(f"    ✓ {orig_count} original images linked/copied")

# Val directory
val_img_dst = Path(DATASET_DIR) / "val" / "images"
val_lbl_dst = Path(DATASET_DIR) / "val" / "labels"
val_img_dst.mkdir(parents=True, exist_ok=True)
val_lbl_dst.mkdir(parents=True, exist_ok=True)

for f in SRC_VAL_IMG.glob("*"):
    dst = val_img_dst / f.name
    if not dst.exists():
        try:
            os.symlink(str(f.resolve()), str(dst))
        except OSError:
            shutil.copy2(str(f), str(dst))

for f in SRC_VAL_LBL.glob("*.txt"):
    dst = val_lbl_dst / f.name
    if not dst.exists():
        try:
            os.symlink(str(f.resolve()), str(dst))
        except OSError:
            shutil.copy2(str(f), str(dst))

# Create augmented images
N_AUGMENTED = 2500
rare_ids = list(bank.keys())
n_created = 0

if rare_ids:
    print(f"\n  Creating {N_AUGMENTED} augmented images...")
    n_created = create_augmented_images(
        SRC_TRAIN_IMG, SRC_TRAIN_LBL, bank, rare_ids,
        combined_train_img, combined_train_lbl, N_AUGMENTED
    )
    print(f"    ✓ Created {n_created} augmented images")

# Verify
final_imgs = list(combined_train_img.glob("*.jpg")) + list(combined_train_img.glob("*.png"))
final_lbls = list(combined_train_lbl.glob("*.txt"))
val_imgs = list(val_img_dst.glob("*"))

new_counts = count_class_instances(combined_train_lbl)
new_total = sum(new_counts.values())

print(f"\n  AFTER augmentation ({new_total} total instances):")
for cid in sorted(new_counts.keys()):
    name = CLASS_NAMES[cid] if cid < NC else f"cls_{cid}"
    old = counts.get(cid, 0)
    new = new_counts[cid]
    added = new - old
    pct = new / new_total * 100
    tag = f" (+{added})" if added > 0 else ""
    print(f"    {cid:2d} {name:20s}: {new:6d} ({pct:5.1f}%){tag}")

print(f"\n  Training : {len(final_imgs)} images, {len(final_lbls)} labels")
print(f"  Validation: {len(val_imgs)} images")

# ── Write dataset.yaml ──
yaml_path = os.path.join(WORK_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path': os.path.abspath(DATASET_DIR),
        'train': 'train/images',
        'val': 'val/images',
        'nc': NC,
        'names': CLASS_NAMES,
    }, f, default_flow_style=False, sort_keys=False)

print(f"  ✓ dataset.yaml → {yaml_path}")
print("=" * 60)

COPY-PASTE AUGMENTATION

  BEFORE augmentation (343205 total instances):
     0 pedestrian          :  79337 ( 23.1%) ██████████████████████████████████████████████
     1 people              :  27059 (  7.9%) ███████████████
     2 bicycle             :  10480 (  3.1%) ██████
     3 car                 : 144867 ( 42.2%) ████████████████████████████████████████████████████████████████████████████████████
     4 van                 :  24956 (  7.3%) ██████████████
     5 truck               :  12875 (  3.8%) ███████
     6 tricycle            :   4812 (  1.4%) ██
     7 awning-tricycle     :   3246 (  0.9%) █
     8 bus                 :   5926 (  1.7%) ███
     9 motor               :  29647 (  8.6%) █████████████████

  Rare classes (< 5%): ['truck', 'bicycle', 'tricycle', 'awning-tricycle', 'bus']
  Building instance bank...
    bicycle             : 425 crops
    truck               : 419 crops
    tricycle            : 424 crops
    awning-tricycle     : 401 crops
    bus          

In [8]:
# ============================================================
# CELL 8: Build model with standard modules, then swap in
#          custom modules. Load pretrained weights. Patch WIoU.
# ============================================================

from ultralytics import YOLO
import importlib

# ─── Step 1: YAML (standard module names for build compat) ───
yaml_content = """
nc: 10
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]          # 0
  - [-1, 1, Conv, [128, 3, 2]]         # 1
  - [-1, 3, C2f, [128]]                # 2  → C2f_DWR
  - [-1, 1, Conv, [256, 3, 2]]         # 3
  - [-1, 6, C2f, [256]]                # 4  → C2f_DWR
  - [-1, 1, Conv, [512, 3, 2]]         # 5
  - [-1, 6, C2f, [512]]                # 6  → C2f_DWR
  - [-1, 1, Conv, [1024, 3, 2]]        # 7
  - [-1, 3, C2f, [1024]]               # 8  → C2f_DWR
  - [-1, 1, SPPF, [1024, 5]]           # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 10
  - [[-1, 6], 1, Concat, [1]]                    # 11 → BiFPNConcat
  - [-1, 3, C2f, [512]]                          # 12 → C2f_FasterBlock

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 13
  - [[-1, 4], 1, Concat, [1]]                    # 14 → BiFPNConcat
  - [-1, 3, C2f, [256]]                          # 15 → C2f_FasterBlock

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 16
  - [[-1, 2], 1, Concat, [1]]                    # 17 → BiFPNConcat
  - [-1, 3, C2f, [128]]                          # 18 → C2f_FasterBlock (P2)

  - [-1, 1, Conv, [128, 3, 2]]                   # 19
  - [[-1, 15], 1, Concat, [1]]                   # 20 → BiFPNConcat
  - [-1, 3, C2f, [256]]                          # 21 → C2f_FasterBlock

  - [-1, 1, Conv, [256, 3, 2]]                   # 22
  - [[-1, 12], 1, Concat, [1]]                   # 23 → BiFPNConcat
  - [-1, 3, C2f, [512]]                          # 24 → C2f_FasterBlock

  - [-1, 1, Conv, [512, 3, 2]]                   # 25
  - [[-1, 9], 1, Concat, [1]]                    # 26 → BiFPNConcat
  - [-1, 3, C2f, [1024]]                         # 27 → C2f_FasterBlock

  - [[18, 21, 24, 27], 1, Detect, [nc]]          # 28
"""

model_yaml_path = os.path.join(WORK_DIR, 'sfd_yolov8_novel.yaml')
with open(model_yaml_path, 'w',encoding='utf-8') as f:
    f.write(yaml_content)
print("✓ YAML saved")

# ─── Step 2: Build model ───
print("\n  Building model with standard C2f/Concat...")
custom_model = YOLO(model_yaml_path)
print("  ✓ Model built!")

# ─── Step 3: Load pretrained YOLOv8n BEFORE replacement ───
print("\n  Loading pretrained YOLOv8n weights...")
try:
    pretrained = YOLO('yolov8n.pt')
    pre_sd = pretrained.model.state_dict()
    cus_sd = custom_model.model.state_dict()

    transferred = 0
    for key in cus_sd:
        if key in pre_sd and cus_sd[key].shape == pre_sd[key].shape:
            cus_sd[key] = pre_sd[key].clone()
            transferred += 1
    custom_model.model.load_state_dict(cus_sd)
    print(f"  ✓ Transferred {transferred}/{len(cus_sd)} pretrained tensors")
    del pretrained
    torch.cuda.empty_cache()
except Exception as e:
    print(f"  ⚠ Pretrained load skipped: {e}")

# ─── Step 4: Swap helper ───
def swap_module(model, idx, new_module, label=""):
    old = model.model.model[idx]
    for attr in ['i', 'f']:
        if hasattr(old, attr):
            setattr(new_module, attr, getattr(old, attr))
    new_module.type = new_module.__class__.__name__
    new_module.np = sum(p.numel() for p in new_module.parameters())
    model.model.model[idx] = new_module
    print(f"    Layer {idx:2d}: {label}")

# ─── Step 5: Backbone C2f → C2f_DWR ───
print("\n  BACKBONE: C2f → C2f_DWR")
for idx in [2, 4, 6, 8]:
    old = custom_model.model.model[idx]
    c1 = old.cv1.conv.in_channels
    c2 = old.cv2.conv.out_channels
    n = len(old.m)
    new = C2f_DWR(c1, c2, n=n)
    try:
        new.cv1.load_state_dict(old.cv1.state_dict())
        new.cv2.load_state_dict(old.cv2.state_dict())
    except:
        pass
    swap_module(custom_model, idx, new, f"C2f({c1}→{c2}, n={n}) → C2f_DWR ✓")

# ─── Step 6: Neck C2f → C2f_FasterBlock ───
print("\n  NECK: C2f → C2f_FasterBlock")
for idx in [12, 15, 18, 21, 24, 27]:
    old = custom_model.model.model[idx]
    c1 = old.cv1.conv.in_channels
    c2 = old.cv2.conv.out_channels
    n = len(old.m)
    new = C2f_FasterBlock(c1, c2, n=n)
    try:
        new.cv1.load_state_dict(old.cv1.state_dict())
        new.cv2.load_state_dict(old.cv2.state_dict())
    except:
        pass
    swap_module(custom_model, idx, new, f"C2f({c1}→{c2}, n={n}) → C2f_FasterBlock ✓")

# ─── Step 7: Concat → BiFPNConcat ───
print("\n  FUSION: Concat → BiFPNConcat")
for idx in [11, 14, 17, 20, 23, 26]:
    new = BiFPNConcat(dimension=1)
    swap_module(custom_model, idx, new, "Concat → BiFPNConcat ✓")

# ─── Step 8: Verify forward pass ───
print("\n  Testing forward pass...")
try:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    test_input = torch.randn(1, 3, 640, 640).to(device)
    custom_model.model.to(device).eval()
    with torch.no_grad():
        _ = custom_model.model(test_input)
    custom_model.model.train()
    custom_model.model.cpu()
    torch.cuda.empty_cache()
    print("  ✓ Forward pass PASSED!")
except Exception as e:
    print(f"  ⚠ Forward pass note: {e}")
    print("    (Training may still work — ultralytics handles device)")

# ─── Step 9: Print architecture ───
print("\n  Final architecture:")
for i, layer in enumerate(custom_model.model.model):
    name = layer.__class__.__name__
    params = sum(p.numel() for p in layer.parameters())
    print(f"    {i:3d}: {name:25s} ({params:,} params)")

total_params = sum(p.numel() for p in custom_model.model.parameters())
print(f"\n  Total parameters: {total_params:,}")

# ─── Step 10: Patch Wise-IoU v3 into ultralytics ───
print("\n  Patching Wise-IoU v3...")

def wiou_v3_bbox_iou(box1, box2, xywh=True, GIoU=False, DIoU=False,
                      CIoU=False, eps=1e-7, **kwargs):
    """
    WIoU v3 — returns quality score.
    Ultralytics computes loss = 1 - quality = focusing × base_loss.
    """
    if xywh:
        b1_x1 = box1[..., 0:1] - box1[..., 2:3] / 2
        b1_y1 = box1[..., 1:2] - box1[..., 3:4] / 2
        b1_x2 = box1[..., 0:1] + box1[..., 2:3] / 2
        b1_y2 = box1[..., 1:2] + box1[..., 3:4] / 2
        b2_x1 = box2[..., 0:1] - box2[..., 2:3] / 2
        b2_y1 = box2[..., 1:2] - box2[..., 3:4] / 2
        b2_x2 = box2[..., 0:1] + box2[..., 2:3] / 2
        b2_y2 = box2[..., 1:2] + box2[..., 3:4] / 2
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)

    w1 = (b1_x2 - b1_x1).clamp(0); h1 = (b1_y2 - b1_y1).clamp(0)
    w2 = (b2_x2 - b2_x1).clamp(0); h2 = (b2_y2 - b2_y1).clamp(0)

    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp_(0) * \
            (b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)).clamp_(0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou = inter / union

    cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)
    ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)
    c2_diag = cw ** 2 + ch ** 2 + eps
    rho2 = ((b1_x1 + b1_x2 - b2_x1 - b2_x2) ** 2 +
            (b1_y1 + b1_y2 - b2_y1 - b2_y2) ** 2) / 4
    rho_w = (w1 - w2) ** 2 / (cw ** 2 + eps)
    rho_h = (h1 - h2) ** 2 / (ch ** 2 + eps)

    cost = rho2 / c2_diag + rho_w + rho_h
    base_loss = 1.0 - iou + cost

    with torch.no_grad():
        mean_bl = base_loss.detach().mean().clamp(min=eps)
        beta = base_loss.detach() / mean_bl
        focusing = (beta * torch.exp(1.0 - beta)).clamp(min=eps, max=2.0)

    # Return quality = 1 - focusing * base_loss
    # So ultralytics loss = 1 - quality = focusing * base_loss  (correct WIoU v3)
    return 1.0 - focusing * base_loss


patched = False
for mod_name in ['ultralytics.utils.metrics', 'ultralytics.utils.loss',
                  'ultralytics.utils.tal']:
    try:
        mod = importlib.import_module(mod_name)
        if hasattr(mod, 'bbox_iou'):
            mod.bbox_iou = wiou_v3_bbox_iou
            print(f"  ✓ Patched {mod_name}.bbox_iou → WIoU v3")
            patched = True
    except:
        pass

if not patched:
    print("  ⚠ WIoU patch not applied — will use default CIoU")

# ─── Summary ───
print("\n" + "=" * 55)
print("  ✅ ALL 6 NOVELTIES READY")
print("=" * 55)
print("  1. ✓ DWR + Coordinate Attention  (backbone)")
print("  2. ✓ BiFPNConcat learned weights  (neck)")
print("  3. ✓ FasterBlock via PConv        (neck)")
print("  4. ✓ 160×160 P2 detection head    (head)")
print("  5. ✓ Wise-IoU v3 loss             (loss)")
print("  6. ✓ Copy-Paste augmentation      (data)")
print(f"  Total params: {total_params:,}")
print("=" * 55)

✓ YAML saved

  Building model with standard C2f/Concat...
WARNING no model scale passed. Assuming scale='n'.
  ✓ Model built!

  Loading pretrained YOLOv8n weights...
  ✓ Transferred 219/437 pretrained tensors

  BACKBONE: C2f → C2f_DWR
    Layer  2: C2f(32→32, n=1) → C2f_DWR ✓
    Layer  4: C2f(64→64, n=2) → C2f_DWR ✓
    Layer  6: C2f(128→128, n=2) → C2f_DWR ✓
    Layer  8: C2f(256→256, n=1) → C2f_DWR ✓

  NECK: C2f → C2f_FasterBlock
    Layer 12: C2f(384→128, n=1) → C2f_FasterBlock ✓
    Layer 15: C2f(192→64, n=1) → C2f_FasterBlock ✓
    Layer 18: C2f(96→32, n=1) → C2f_FasterBlock ✓
    Layer 21: C2f(96→64, n=1) → C2f_FasterBlock ✓
    Layer 24: C2f(192→128, n=1) → C2f_FasterBlock ✓
    Layer 27: C2f(384→256, n=1) → C2f_FasterBlock ✓

  FUSION: Concat → BiFPNConcat
    Layer 11: Concat → BiFPNConcat ✓
    Layer 14: Concat → BiFPNConcat ✓
    Layer 17: Concat → BiFPNConcat ✓
    Layer 20: Concat → BiFPNConcat ✓
    Layer 23: Concat → BiFPNConcat ✓
    Layer 26: Concat → BiFPNConcat 

In [9]:
# ============================================================
# CELL 9: Training — 300 Epochs with Checkpoint Saving
#
# KEY FEATURES:
#   • save_period=10  → saves checkpoint every 10 epochs
#   • patience=50     → early stop only after 50 stale epochs
#   • cos_lr=True     → cosine annealing LR (smoother decay)
#   • amp=True        → mixed precision (saves VRAM)
#   • If interrupted, run CELL 10 to resume from last.pt
# ============================================================

CONFIG = {
    'epochs':         300,
    'batch':          16,         # adjust based on your GPU VRAM
    'imgsz':          640,
    'lr0':            0.01,
    'lrf':            0.01,
    'momentum':       0.937,
    'weight_decay':   0.0005,
    'optimizer':      'SGD',
    'device':         0,
    'workers':        4,
    'patience':       50,
    'mosaic':         1.0,
    'close_mosaic':   25,        # disable mosaic last 25 epochs
    'save_period':    10,        # ★ CHECKPOINT EVERY 10 EPOCHS
    'project':        WORK_DIR,
    'name':           'train_6novelties_300ep',
    'exist_ok':       True,
    'verbose':        True,
    'save':           True,
    'plots':          True,
    'cos_lr':         True,      # ★ cosine LR schedule
    'amp':            True,      # ★ mixed precision
    'label_smoothing': 0.005,    # ★ slight label smoothing
}

TRAIN_DIR = os.path.join(CONFIG['project'], CONFIG['name'])
print("=" * 60)
print("SFD-YOLOv8 NOVEL — TRAINING (300 EPOCHS)")
print("=" * 60)
print(f"""
  NOVELTIES ACTIVE:
    1. DWR + Coordinate Attention    (backbone)
    2. BiFPNConcat learned weights   (neck fusion)
    3. FasterBlock via PConv         (neck)
    4. 160×160 P2 detection head     (small targets)
    5. Wise-IoU v3 loss              (dynamic focusing)
    6. Copy-Paste augmentation       (rare classes, +{n_created} images)

  ACCURACY IMPROVEMENTS:
    ✓ 300 epochs (vs 200)
    ✓ Cosine LR schedule             (smoother convergence)
    ✓ Label smoothing 0.005          (regularization)
    ✓ Mixed precision training       (faster + memory efficient)
    ✓ Checkpoint every 10 epochs     (resume if interrupted)
    ✓ Patience 50 epochs             (longer exploration)
    ✓ Close mosaic at epoch 275      (fine-tune last 25 epochs)
    ✓ More copy-paste augmentation   (2500 images)

  Training: {len(final_imgs)} images | Val: {len(val_imgs)} images
  Epochs: {CONFIG['epochs']} | Batch: {CONFIG['batch']} | ImgSz: {CONFIG['imgsz']}
  Checkpoints saved to: {TRAIN_DIR}/weights/
""")

results = custom_model.train(
    data=yaml_path,
    epochs=CONFIG['epochs'],
    batch=CONFIG['batch'],
    imgsz=CONFIG['imgsz'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    optimizer=CONFIG['optimizer'],
    device=CONFIG['device'],
    workers=CONFIG['workers'],
    patience=CONFIG['patience'],
    mosaic=CONFIG['mosaic'],
    close_mosaic=CONFIG['close_mosaic'],
    save_period=CONFIG['save_period'],
    project=CONFIG['project'],
    name=CONFIG['name'],
    exist_ok=CONFIG['exist_ok'],
    verbose=CONFIG['verbose'],
    save=CONFIG['save'],
    plots=CONFIG['plots'],
    cos_lr=CONFIG['cos_lr'],
    amp=CONFIG['amp'],
    label_smoothing=CONFIG['label_smoothing'],
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mixup=0.15,
    copy_paste=0.1,
)

print(f"\n{'=' * 60}")
print("  ✅ TRAINING COMPLETE!")
print(f"{'=' * 60}")
print(f"  Best model: {TRAIN_DIR}/weights/best.pt")
print(f"  Last model: {TRAIN_DIR}/weights/last.pt")

SFD-YOLOv8 NOVEL — TRAINING (300 EPOCHS)

  NOVELTIES ACTIVE:
    1. DWR + Coordinate Attention    (backbone)
    2. BiFPNConcat learned weights   (neck fusion)
    3. FasterBlock via PConv         (neck)
    4. 160×160 P2 detection head     (small targets)
    5. Wise-IoU v3 loss              (dynamic focusing)
    6. Copy-Paste augmentation       (rare classes, +2500 images)

  ACCURACY IMPROVEMENTS:
    ✓ 300 epochs (vs 200)
    ✓ Cosine LR schedule             (smoother convergence)
    ✓ Label smoothing 0.005          (regularization)
    ✓ Mixed precision training       (faster + memory efficient)
    ✓ Checkpoint every 10 epochs     (resume if interrupted)
    ✓ Patience 50 epochs             (longer exploration)
    ✓ Close mosaic at epoch 275      (fine-tune last 25 epochs)
    ✓ More copy-paste augmentation   (2500 images)

  Training: 8971 images | Val: 548 images
  Epochs: 300 | Batch: 16 | ImgSz: 640
  Checkpoints saved to: C:\Users\CSE_SDPL\sfd_yolov8_novel\train_6novelti

In [15]:
# ============================================================
# CELL 11: Validation + Results Visualization
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

print("=" * 60)
print("VALIDATION & RESULTS")
print("=" * 60)

WORK_DIR = os.path.join(os.path.expanduser("~"), "sfd_yolov8_novel")

# ── Find best.pt ──
candidates = sorted(Path(WORK_DIR).rglob("best.pt"))
if not candidates:
    raise FileNotFoundError(f"No best.pt found under {WORK_DIR}")

best_pt = str(candidates[-1])
RESULTS_DIR = candidates[-1].parents[1]
print(f"  ✓ Best model: {best_pt}")
print(f"  ✓ Results dir: {RESULTS_DIR}")

# ── Find dataset.yaml ──
yaml_path = os.path.join(WORK_DIR, 'dataset.yaml')
if not os.path.exists(yaml_path):
    yaml_candidates = sorted(Path(WORK_DIR).rglob("dataset.yaml"))
    if yaml_candidates:
        yaml_path = str(yaml_candidates[-1])
print(f"  ✓ Dataset YAML: {yaml_path}")

device = 0 if torch.cuda.is_available() else 'cpu'

# ── Run validation ──
print("\n  Running validation on best model...")
val_model = YOLO(best_pt)
metrics = val_model.val(
    data=yaml_path,
    imgsz=640,
    batch=8,
    device=device,
    verbose=True,
    plots=True,
)

# ── Print metrics ──
print(f"\n{'=' * 60}")
print("  VALIDATION RESULTS")
print(f"{'=' * 60}")
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")

CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]

print(f"\n  Per-class AP50:")
if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
    ap50 = metrics.box.ap50
    if hasattr(ap50, '__len__'):
        for i, ap in enumerate(ap50):
            name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"cls_{i}"
            bar = "█" * int(float(ap) * 40)
            print(f"    {name:20s}: {float(ap):.4f}  {bar}")

# ── Display training plots ──
plot_names = [
    "results.png", "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "F1_curve.png", "P_curve.png", "R_curve.png", "PR_curve.png",
    "labels.jpg", "labels_correlogram.jpg",
]

found_plots = []
for pname in plot_names:
    p = RESULTS_DIR / pname
    if p.exists():
        found_plots.append(p)

if found_plots:
    n = len(found_plots)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows))
    if rows * cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, ppath in enumerate(found_plots):
        img = mpimg.imread(str(ppath))
        axes[i].imshow(img)
        axes[i].set_title(ppath.name, fontsize=10)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / "all_results_summary.png"), dpi=150)
    plt.show()
    print(f"\n  ✓ Summary plot saved to: {RESULTS_DIR / 'all_results_summary.png'}")

print(f"\n{'=' * 60}")
print("  ✅ VALIDATION COMPLETE")
print(f"{'=' * 60}")

VALIDATION & RESULTS
  ✓ Best model: C:\Users\CSE_SDPL\sfd_yolov8_novel\train_6novelties_300ep\weights\best.pt
  ✓ Results dir: C:\Users\CSE_SDPL\sfd_yolov8_novel\train_6novelties_300ep
  ✓ Dataset YAML: C:\Users\CSE_SDPL\sfd_yolov8_novel\dataset.yaml

  Running validation on best model...
Ultralytics 8.4.21  Python-3.10.19 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
sfd_YOLOv8_novel summary (fused): 91 layers, 2,922,360 parameters, 0 gradients, 12.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 463.8153.2 MB/s, size: 126.7 KB)
val: Scanning C:\Users\CSE_SDPL\sfd_yolov8_novel\dataset\val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 9.5it/s 7.2s<0.1s
                   all        548      38759      0.487      0.359      0.374      0.225
            pedestrian        520       8844      0.577      0.36

<Figure size 2400x1200 with 6 Axes>


  ✓ Summary plot saved to: C:\Users\CSE_SDPL\sfd_yolov8_novel\train_6novelties_300ep\all_results_summary.png

  ✅ VALIDATION COMPLETE


In [21]:
# Run on VAL to reproduce the paper's 35.3%
metrics_val = model.val(
    data     = str(yaml_path),
    split    = "val",          # ← paper likely used this
    imgsz    = 640,
    conf     = 0.001,          # low conf for proper mAP
    iou      = 0.6,
    device   = 0,
    project  = WORK_DIR,
    name     = "eval_val_reproduce",
    exist_ok = True,
    plots    = True,
)

print(f"\n   mAP@0.5      = {metrics_val.box.map50:.4f}")
print(f"   mAP@0.5:0.95 = {metrics_val.box.map:.4f}")
print(f"   Precision    = {metrics_val.box.mp:.4f}")
print(f"   Recall       = {metrics_val.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.10.19 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 175.359.2 MB/s, size: 93.6 KB)
val: Scanning C:\Users\CSE_SDPL\sfd_yolov8_novel\dataset\val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.8it/s 9.3s0.2s
                   all        548      38759      0.492       0.37      0.383      0.227
            pedestrian        520       8844      0.596      0.383      0.448      0.204
                people        482       5125      0.572      0.319      0.373      0.146
               bicycle        364       1287      0.284      0.117      0.119      0.048
                   car        515      14064      0.708      0.795      0.814      0.571
                   van        421       1975      0.528      0.413      0.434      0.307
      